# 🌍 UNDP SDG Analytics — Exploratory Data Analysis

This notebook walks through the full analytical pipeline:
1. Data loading & cleaning
2. Exploratory statistics
3. K-Means clustering with validation
3b. Sensitivity analysis (k=2–6) — testing k=3 against alternatives
4. Gini coefficient
5. HDI trend analysis

**Data sources:** UNDP HDR 2023 · World Bank WDI · Our World in Data

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from utils.data_loader import load_main, load_trends, attach_cluster_labels, FEATURES
from utils.clustering import kmeans, gini_coefficient, elbow_curve, silhouette_score_manual

plt.style.use('dark_background')
pd.set_option('display.max_columns', None)
print('Libraries loaded ✓')

## 1. Load & Clean Data

In [ ]:
df = load_main('../data/sdg_indicators.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Missing value audit
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nImputed rows: {df["_imputed"].sum()}')

In [ ]:
# Descriptive statistics
df[FEATURES + ['gni_per_capita', 'literacy_rate']].describe().round(3)

## 2. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr = df[['hdi', 'poverty_rate', 'literacy_rate',
           'internet_access', 'gni_per_capita', 'co2_per_capita']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax)
ax.set_title('SDG Indicator Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig('../screenshots/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. K-Means Clustering

In [ ]:
# Elbow Method
wcss_results = elbow_curve(df, FEATURES, k_range=range(2, 9))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(wcss_results.keys()), list(wcss_results.values()),
        marker='o', color='#00D4FF', linewidth=2.5, markersize=8)
ax.axvline(x=3, color='#34D399', linestyle='--', alpha=0.7, label='Chosen k=3')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('WCSS')
ax.set_title('Elbow Method — Optimal k Selection')
ax.legend()
plt.tight_layout()
plt.savefig('../screenshots/elbow_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Elbow results:', wcss_results)

In [ ]:
# Run K-Means with k=3
labels = kmeans(df, FEATURES, k=3)
df = attach_cluster_labels(df, labels)

# Silhouette Score
sil = silhouette_score_manual(df, FEATURES, df['cluster_id'])
print(f'Silhouette Score: {sil:.3f}  (>0.5 = good separation)')

# Countries per cluster
for cluster in df['cluster'].unique():
    members = df[df['cluster'] == cluster]['country'].tolist()
    print(f'\n{cluster}: {members}')

## 3b. Sensitivity Analysis: Is k=3 the Best Choice?

K-Means requires choosing k in advance. Instead of assuming k=3 is
correct because the elbow plot suggested it, this section checks the
silhouette score across k=2 through k=6 directly — including k=4 as a
genuine alternative — rather than presenting k=3 as automatically optimal.

In [ ]:
from sklearn.metrics import silhouette_score as sk_silhouette

sensitivity_results = []
label_sets = {}
df_norm = df.copy()
from utils.clustering import normalise
X_norm = normalise(df, FEATURES)[FEATURES].values

for k in range(2, 7):
    lbls = kmeans(df, FEATURES, k=k)
    sil = sk_silhouette(X_norm, lbls)
    sensitivity_results.append({'k': k, 'silhouette_score': round(sil, 3)})
    label_sets[k] = lbls

sens_df = pd.DataFrame(sensitivity_results)
sens_df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bar_colors = ['#00D4FF' if k == 3 else ('#FBBF24' if k == 4 else '#475569')
              for k in sens_df['k']]
ax.bar(sens_df['k'].astype(str), sens_df['silhouette_score'], color=bar_colors)
for i, v in enumerate(sens_df['silhouette_score']):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center')
ax.set_xlabel('k')
ax.set_ylabel('Silhouette Score')
ax.set_title('Sensitivity Analysis: Silhouette Score by k')
plt.tight_layout()
plt.show()

print(f"Best silhouette score is at k={sens_df.loc[sens_df.silhouette_score.idxmax(), 'k']}, "
      f"NOT k=3. k=3 is kept as the primary model anyway because it maps "
      f"onto UNDP's Low/Medium/High classification -- interpretability, "
      f"not just silhouette score, drove the final choice.")

**Reading this honestly:** k=3 has the *second-lowest* silhouette score of
the range tested (0.460) — k=5 scores highest (0.505). The differences
are all fairly small (0.46–0.51), so no single k is decisively correct on
separation quality alone. Given that, k=3 is kept as the primary model
for interpretability — it aligns with UNDP's own Low/Medium/High human
development framework — not because the statistics forced that choice.

In [ ]:
# What actually changes at k=4 vs k=3?
k4_labels = label_sets[4]
compare = df[['country', 'cluster_id', 'cluster']].copy()
compare['k4_raw'] = k4_labels.values

hdi_by_k4 = df.assign(k4=k4_labels.values).groupby('k4')['hdi'].mean().sort_values()
remap4 = {old: new for new, old in enumerate(hdi_by_k4.index)}
compare['k4_cluster'] = compare['k4_raw'].map(remap4)

# Which k=3 cluster maps to more than one k=4 cluster? That's the one that split.
split_check = compare.groupby('cluster')['k4_cluster'].nunique()
print('Number of distinct k=4 sub-clusters per k=3 cluster:')
print(split_check)
compare[['country', 'cluster', 'k4_cluster']].sort_values(['cluster', 'k4_cluster'])

In [ ]:
# Scatter: HDI vs Internet, coloured by cluster
colors = {'Vulnerable': '#F87171', 'Developing': '#FBBF24', 'Advanced': '#34D399'}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for cluster, grp in df.groupby('cluster'):
    axes[0].scatter(grp['internet_access'], grp['hdi'],
                    c=colors[cluster], label=cluster, s=80, alpha=0.85)
    axes[1].scatter(grp['poverty_rate'], grp['hdi'],
                    c=colors[cluster], label=cluster, s=80, alpha=0.85)

axes[0].set_xlabel('Internet Access (%)'); axes[0].set_ylabel('HDI')
axes[0].set_title('HDI vs Internet Access'); axes[0].legend()
axes[1].set_xlabel('Poverty Rate (%)'); axes[1].set_ylabel('HDI')
axes[1].set_title('HDI vs Poverty Rate'); axes[1].legend()

plt.tight_layout()
plt.savefig('../screenshots/clustering_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Gini Coefficient — Income Inequality

In [ ]:
gini = gini_coefficient(df['gni_per_capita'])
print(f'Gini Coefficient (GNI/capita): {gini:.3f}')
print('Interpretation: 0 = perfect equality · 1 = perfect inequality')

# Lorenz Curve
sorted_gni = np.sort(df['gni_per_capita'].values)
n = len(sorted_gni)
lorenz_x = np.linspace(0, 1, n)
lorenz_y = np.cumsum(sorted_gni) / sorted_gni.sum()

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(lorenz_x, lorenz_y, color='#00D4FF', linewidth=2.5, label='Lorenz Curve')
ax.plot([0, 1], [0, 1], '--', color='#64748B', label='Perfect Equality')
ax.fill_between(lorenz_x, lorenz_y, lorenz_x, alpha=0.15, color='#F87171')
ax.set_xlabel('Cumulative Share of Countries')
ax.set_ylabel('Cumulative Share of GNI')
ax.set_title(f'Lorenz Curve — Gini = {gini:.3f}')
ax.legend()
plt.tight_layout()
plt.savefig('../screenshots/lorenz_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. HDI Trend Analysis (2015–2023)

In [ ]:
df_trend = load_trends('../data/hdi_trends.csv')

fig, ax = plt.subplots(figsize=(10, 5))
palette = plt.cm.Set2.colors

for i, (country, grp) in enumerate(df_trend.groupby('country')):
    grp = grp.sort_values('year')
    ax.plot(grp['year'], grp['hdi'], marker='o',
            color=palette[i % len(palette)], linewidth=2.2,
            markersize=7, label=country)

ax.axvspan(2020, 2022, alpha=0.08, color='#F87171', label='COVID-19 period')
ax.set_xlabel('Year'); ax.set_ylabel('HDI')
ax.set_title('HDI Trends (2015–2023) — COVID-19 Regression Visible in 2021')
ax.legend(loc='lower right', fontsize=9)
ax.set_xticks([2015, 2017, 2019, 2021, 2023])
plt.tight_layout()
plt.savefig('../screenshots/hdi_trends.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# HDI change 2015→2023
pivot = df_trend.pivot(index='country', columns='year', values='hdi')
pivot['change_2015_2023'] = pivot[2023] - pivot[2015]
print(pivot[['change_2015_2023']].sort_values('change_2015_2023', ascending=False))